# 🧾 生成式AI應用開發｜第06週
## Structured Outputs、JSON Schema 與資料抽取

> **執行環境：Google Colab** ｜ **主線：OpenAI Responses API**

本週把第 4 週 Prompt Engineering 和第 5 週 API 狀態管理，推進到「可被程式穩定處理」的結構化輸出。核心目標不是讓模型回答得漂亮，而是讓模型輸出能直接變成 Python dict、資料表、表單欄位或後續 App 的輸入。

請用三層來閱讀本章：

1. **輸入文字**：原本散亂的自然語言，例如活動通知、作業說明、客服訊息。
2. **輸出規格**：JSON Schema 或 Pydantic model，規定最後必須有哪些欄位、欄位型別是什麼。
3. **程式檢查**：Python 取得結果後，再確認欄位是否能被 App 安全使用。

如果一開始看不懂 `schema`，先把它想成「資料表欄位規格」。例如 `title`、`date`、`owner` 就是之後程式會讀取的欄位名稱。

官方文件入口：
- Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- Responses API: https://platform.openai.com/docs/api-reference/responses

> **本週任務**
> 使用 JSON Schema 約束模型輸出，完成課程事件、作業清單與客服資料抽取。


> **學生版**：保留課堂練習 TODO。請先完成練習 A，再依時間完成練習 B 與選做練習 C。


## 🎯 本週學習目標

| # | 能力 | 後續應用 |
|---|------|----------|
| 1 | 分辨一般 JSON prompt 與 Structured Outputs | 降低輸出格式漂移 |
| 2 | 撰寫基本 JSON Schema | 表單、資料抽取、資料庫欄位 |
| 3 | 使用 Responses API 的 `text.format` | 讓模型遵守指定 schema |
| 4 | 將 `response.output_text` 轉成 Python dict | 串接 App 邏輯 |
| 5 | 驗證必填欄位與型別 | 防止錯誤資料進入系統 |
| 6 | 批次測試資料抽取結果 | 建立可維護的 Prompt / Schema |
| 7 | 設計自己的抽取器 | 期中、期末專題資料處理 |

本週最重要的觀念是：Structured Outputs 不是「讓模型更聰明」，而是「讓模型的輸出更像程式可以信任的資料」。因此讀程式時請先問兩件事：

- 這個欄位之後會被哪一段程式使用？
- 如果欄位漏掉、型別錯誤或多出欄位，App 會不會壞掉？


## 🗺️ 三堂課（150 分鐘）實作流程

| 時間 | 主題 | 產出 |
|------|------|------|
| 0-15 分 | 非結構化輸出的問題 | 看懂為什麼只靠 prompt 不夠 |
| 15-35 分 | JSON Schema 基礎 | 寫出第一個 event schema |
| 35-60 分 | Structured Outputs API | 取得可解析的 JSON 字串 |
| 60-80 分 | Python 驗證與錯誤處理 | 建立 `validate_required_fields()` |
| 80-105 分 | 陣列與巢狀資料抽取 | 作業清單抽取器 |
| 105-130 分 | 批次測試與 schema 版本 | 比較多筆輸入是否穩定 |
| 130-150 分 | 三組課堂練習 | 完成自己的抽取器 |

閱讀順序建議：先跑不用 API 的 cell，看懂 schema 長相；再決定是否把 `RUN_...` 改成 `True` 執行付費 API cell。遇到看不懂的巢狀結構時，先只看最外層 `type`、`properties`、`required`，再往內看 `items`。


---
## 0. 環境準備

本週沿用第 3-5 週的 OpenAI API 設定方式。請先確認 Colab Secrets 或本機環境變數中已有 `OPENAI_API_KEY`。


In [ ]:
# 安裝或更新 OpenAI Python SDK
# Pydantic 用於進階 structured parsing 與欄位驗證。
!pip install -q --upgrade openai pydantic


In [ ]:
import json
import os
from copy import deepcopy
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

# 這個 setup cell 會被後面所有示範共用；如果這裡失敗，後面的 API、schema、Pydantic 範例都不應繼續執行。
# 把環境檢查集中放在一開始，可以讓學生先解決 key / SDK / model 問題，再進入資料抽取主題。

# 本週會把模型輸出轉成 JSON / Pydantic 物件，所以先準備三類工具：
# 1. json：把 JSON 字串轉成 Python dict/list。
# 2. Pydantic：用 class 描述資料欄位與驗證規則。
# 3. OpenAI client：呼叫 Responses API。

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。
# 這段只負責取得 API key，不會顯示 key 的內容。
try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 不在 Colab 執行時，google.colab 不存在，直接略過即可。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")

if not os.getenv("OPENAI_API_KEY"):
    # 這裡故意直接中止：沒有 key 時若繼續往下跑，錯誤會出現在較遠的 API cell，學生比較難定位問題。
    raise RuntimeError("找不到 OPENAI_API_KEY。請先設定 Colab Secret 或本機環境變數。")

# OPENAI_MODEL 可讓老師課前改成實際可用模型，不必改後面每一個 API 呼叫。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
# OpenAI() 會從環境變數讀取 OPENAI_API_KEY；不要把 key 寫在 notebook 裡，避免上傳或分享時外洩。
client = OpenAI()
print("本週模型：", MODEL)
print("API key 已設定（不顯示內容）")


In [ ]:
STRUCTURED_INSTRUCTIONS = """
你是資料抽取助理。請只根據使用者提供的文字抽取資料。
如果資料沒有出現，請使用 null 或空陣列，不要自行補充。
"""


def structured_response(user_input, schema, name="structured_output", instructions=STRUCTURED_INSTRUCTIONS):
    """
    使用 Responses API 與 JSON Schema 取得結構化輸出回應。

    這是本週 JSON Schema 主線的共同入口。呼叫端提供一段自然語言文字，
    再提供一份描述輸出欄位的 schema；API 會依照 schema 產生可被程式解析的 JSON。
    這個函式刻意只負責「送出請求並拿回 response」，不在這裡解析 JSON，
    讓學生能清楚分辨 API 呼叫、拒答檢查、JSON 解析與欄位驗證是四個不同步驟。

    參數 (Args):
        user_input:   要抽取資料的原始文字，必須是非空白字串。
        schema:       JSON Schema dict，最外層必須是 object。
        name:         這份輸出格式的識別名稱，方便除錯與管理多個 schema。
        instructions: 給模型的任務說明，用來限制抽取角色與資料來源。

    回傳 (Returns):
        OpenAI Responses API 的 response 物件；後續再由 parse_output_text() 解析。
    """

    # 這個 helper 的責任很窄：只把「自然語言輸入 + schema」送到 API，並回傳原始 response。
    # 解析 JSON、處理拒答、驗證欄位會放在後面的函式，讓每一層錯誤比較容易說明與測試。
    # 情況一：輸入文字不合法
    # 先在本機檢查輸入，比讓 API 收到空白文字後回傳模糊錯誤更適合教學與除錯。
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")

    # 情況二：schema 格式不符合本週範例要求
    # 本週範例都要求最外層是 object，因為 App 後續通常會用固定欄位名稱讀取 dict。
    if not isinstance(schema, dict) or schema.get("type") != "object":
        raise ValueError("schema 必須是 JSON Schema object。")

    # 這裡是本週最重要的 API 參數：
    # - input：原始自然語言文字。
    # - text.format.type = json_schema：要求模型依照 JSON Schema 輸出。
    # - schema：你在 Python dict 中寫好的欄位規格。
    # - strict=True：要求模型更嚴格遵守 schema，不要多生未允許欄位。
    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=user_input,
        text={
            "format": {
                "type": "json_schema",
                # name 是這份輸出格式的識別名稱；在除錯或日後管理多個 schema 時很有用。
                "name": name,
                # schema 是模型必須遵守的資料契約；欄位、型別、required 都從這裡來。
                "schema": schema,
                # strict=True 讓模型輸出更貼近 schema；正式 App 仍要保留後面的程式端檢查。
                "strict": True,
            }
        },
    )
    # 這裡不直接回傳 dict，是為了讓後面可以示範 response.output_text、refusal 與 parsing 的分工。
    return response


def _read_attr_or_key(obj, name, default=None):
    """
    同時支援「SDK 物件」與「dict（字典）」兩種資料型態的通用讀取工具。

    在處理 API 回應時，回傳內容有時是 SDK 封裝好的物件（用 obj.name 存取），
    有時則是解析後或測試用的 dict（用 obj["name"] 存取）。這個函式把兩種情況統一起來，
    讓呼叫端不必先判斷型態，就能安全地取出想要的欄位值。

    參數 (Args):
        obj:     要讀取的來源，可以是 dict 或任意 SDK 物件。
        name:    想要讀取的欄位名稱 / 屬性名稱（字串）。
        default: 當該欄位或屬性不存在時要回傳的預設值，預設為 None。

    回傳 (Returns):
        取到的欄位值；若不存在，則回傳 default。
    """

    # 情況一：obj 是字典（dict）
    # 使用 dict.get(name, default)，找不到 key 時不會拋出例外，而是安全地回傳 default。
    if isinstance(obj, dict):
        return obj.get(name, default)

    # 情況二：obj 是一般物件（例如 SDK 回傳的物件）
    # 使用 getattr(obj, name, default) 讀取屬性，若屬性不存在同樣回傳 default，避免 AttributeError。
    return getattr(obj, name, default)


def get_refusal_reason(response):
    """
    從 Structured Outputs 回應中找出模型是否拒答，以及拒答原因。

    Structured Outputs 讓一般成功輸出遵守 schema，但安全拒答是例外情況。
    當模型拒答時，response 可能沒有符合 schema 的 output_text，而是在 output/content 裡放 refusal。
    因此在 json.loads() 前先檢查 refusal，可以避免把拒答訊息誤當成正式資料寫入 App。

    參數 (Args):
        response: OpenAI Responses API 回傳的 response 物件，或測試時模擬的 dict。

    回傳 (Returns):
        若找到拒答原因，回傳 refusal 文字；若沒有拒答，回傳 None。
    """

    # 情況一：response.output 可能不存在或是空值
    # 使用 _read_attr_or_key(..., []) or [] 可以讓迴圈在沒有 output 時安全略過。
    for output_item in _read_attr_or_key(response, "output", []) or []:
        # 情況二：每個 output item 裡可能有多個 content item
        # Structured Outputs 的拒答資訊通常藏在 content item 的 refusal 欄位。
        for content_item in _read_attr_or_key(output_item, "content", []) or []:
            refusal = _read_attr_or_key(content_item, "refusal")
            if refusal:
                # 只要找到 refusal，就代表不能把這次回應當成符合 schema 的資料使用。
                return refusal

    # 沒找到 refusal，不代表資料一定正確；只代表可以進入下一步 output_text 檢查與 JSON 解析。
    return None


def parse_output_text(response):
    """
    將 Structured Outputs 的 output_text 從 JSON 字串轉成 Python dict。

    API 回應回來後，程式不能立刻假設它就是可用資料。這個函式把解析前的防守流程集中起來：
    先檢查模型是否拒答，再檢查 output_text 是否存在，最後才用 json.loads() 轉成 dict。
    這種順序適合正式 App，因為它能把「模型拒答」「沒有輸出」「JSON 格式錯誤」分成不同錯誤來源。

    參數 (Args):
        response: structured_response() 回傳的 Responses API response 物件。

    回傳 (Returns):
        解析後的 Python dict，欄位應符合呼叫 structured_response() 時提供的 schema。

    可能拋出 (Raises):
        ValueError: 模型拒答或 response 沒有 output_text。
        json.JSONDecodeError: output_text 不是合法 JSON 字串。
    """

    # 步驟一：先檢查拒答
    # 如果模型拒答，回應通常不會符合原本 schema，應該回報錯誤而不是繼續解析。
    refusal = get_refusal_reason(response)
    if refusal:
        raise ValueError(f"模型拒答，未產生符合 schema 的資料：{refusal}")

    # 步驟二：再檢查 output_text 是否存在
    # Structured Outputs 成功時應該有 output_text；空值通常表示回應結構、模型拒答或 SDK 行為需要另外檢查。
    output_text = getattr(response, "output_text", "")
    if not output_text:
        raise ValueError("回應沒有 output_text，請檢查 API 回應或模型是否拒答。")

    # 步驟三：最後才做 JSON 解析
    # 到這一步才把 JSON 字串轉成 dict；前面的防守檢查可以避免把拒答文字誤當成資料。
    return json.loads(output_text)


def print_json(data, title=None):
    """
    以容易閱讀的縮排格式顯示 dict 或 list。

    課堂上直接 print(dict) 雖然可行，但巢狀 JSON 會擠成一行，不利於觀察欄位層次。
    這個函式統一使用 ensure_ascii=False 保留繁體中文，並用 indent=2 顯示巢狀結構。

    參數 (Args):
        data:  要顯示的 dict 或 list。
        title: 選填標題，方便在同一個 cell 中區分多份輸出。

    回傳 (Returns):
        無。這個函式只負責印出格式化結果。
    """

    # 情況一：有傳入 title 時，先印出段落標題，方便課堂截圖或比較多次輸出。
    if title:
        print(f"=== {title} ===")

    # 情況二：直接顯示 JSON 內容
    # ensure_ascii=False 可以保留中文字；indent=2 讓 object / array 的層次更清楚。
    print(json.dumps(data, ensure_ascii=False, indent=2))


> **成本提醒**
> 本週多數 API cell 都會產生付費請求。批次測試與延伸實驗預設關閉，確認額度後再把 `RUN_...` 變數改為 `True`。


---
## 1. 為什麼不能只叫模型「請輸出 JSON」？

Prompt 可以要求模型輸出 JSON，但模型仍可能多加說明文字、漏掉欄位、改欄位名稱，或把數字輸出成文字。Structured Outputs 的目標，是讓輸出符合你提供的 JSON Schema。

請把兩種做法分開：

- **一般 JSON prompt**：你用文字拜託模型照格式輸出，模型可能遵守，也可能偏離。
- **Structured Outputs**：你把 schema 放進 API 參數，API 會要求模型輸出符合該 schema 的 JSON。

這一節先故意示範一般 prompt，目的是觀察「看起來像 JSON」和「程式可穩定解析」之間的差距。


In [ ]:
loose_prompt = """
請把以下文字整理成 JSON：
下週三 14:00 在 A203 教室有 AI 專題討論，負責人是小安。
"""

# 這個 cell 故意不使用 schema，只用一般 prompt。
# 請先觀察模型可能輸出哪些不穩定格式，再和後面的 Structured Outputs 比較。
# 這個開關讓老師可以在課堂上控制是否真的花費 API 額度；預設 False 也方便學生先閱讀程式。
RUN_LOOSE_JSON_DEMO = False
if RUN_LOOSE_JSON_DEMO:
    loose = client.responses.create(
        model=MODEL,
        instructions="你是資料整理助理，請用繁體中文回答。",
        input=loose_prompt,
    )
    # 這裡直接印原文，是為了觀察一般 prompt 可能出現 markdown code block、欄位命名飄移或額外解釋。
    print(loose.output_text)
else:
    print("一般 JSON prompt 示範尚未執行；確認成本後可將 RUN_LOOSE_JSON_DEMO 改為 True。")


### ✍️ 觀察紀錄

執行上方非結構化版本時，請觀察：

- 是否有多餘說明文字？
- 欄位名稱是否固定？
- 日期、時間、地點是否容易被穩定取出？
- 如果要交給程式處理，還需要做哪些清理？


---
## 2. JSON Schema 的基本組成

| Schema 欄位 | 用途 | 可以怎麼想 |
|-------------|------|------------|
| `type` | 指定資料型態，例如 `object`、`string`、`array` | 這一層資料是物件、文字、數字或清單 |
| `properties` | 定義 object 內有哪些欄位 | 資料表有哪些欄位 |
| `required` | 指定一定要出現的欄位 | App 後面一定會讀到的欄位 |
| `additionalProperties` | 是否允許未列出的欄位 | 是否允許模型自行新增欄位 |

在 `strict=True` 時，建議清楚列出 `required`，並設定 `additionalProperties: False`。

閱讀 schema 的順序：

1. 先看最外層 `type`，確認最後輸出是一個 object 還是一個 array。
2. 再看 `properties`，知道會有哪些欄位。
3. 接著看每個欄位自己的 `type`，例如 `string`、`boolean`、`array`。
4. 最後看 `required` 與 `additionalProperties`，判斷程式端能不能穩定使用。


In [ ]:
# event_schema 是本週第一個正式輸出規格。
# 最外層 type=object，代表最後輸出會是一個 Python dict。
event_schema = {
    "type": "object",
    "properties": {
        # properties 裡的每個 key，都是之後程式會讀取的欄位名稱。
        # title 不允許 null，因為沒有事件名稱時，這筆資料通常不適合進入後續行事曆或任務系統。
        "title": {"type": "string", "description": "事件名稱"},
        # ["string", "null"] 表示可以是文字，也可以是 null。
        # 當原文沒有提供日期、時間、地點或負責人時，用 null 比讓模型猜測更安全。
        "date": {"type": ["string", "null"], "description": "日期，保留原文格式"},
        "time": {"type": ["string", "null"], "description": "時間，保留原文格式"},
        # location / owner 常常在原文缺漏，所以也允許 null；App 可在畫面上提醒使用者補齊。
        "location": {"type": ["string", "null"], "description": "地點"},
        "owner": {"type": ["string", "null"], "description": "負責人"},
    },
    # required 是「欄位一定要出現」，不是「欄位一定有值」。
    # 因為 date/time/location/owner 允許 null，所以即使資訊未知，欄位仍會保留。
    "required": ["title", "date", "time", "location", "owner"],
    # False 表示不允許模型自行新增 schema 沒寫的欄位，讓後續程式更好處理。
    "additionalProperties": False,
}

print_json(event_schema, title="event_schema")


### 🧱 Schema 設計原則

欄位名稱要清楚、穩定，並接近程式會使用的資料名稱。欄位描述不是給學生看的註解，而是提供模型判斷欄位意義的線索。

本例的 `event_schema` 可以拆成這樣看：

- `title`：事件名稱，後續可以顯示在行事曆或表格標題。
- `date`、`time`：先保留原文格式，不急著轉成標準日期時間。
- `location`：地點可能是教室、線上會議或未提供，所以允許 `null`。
- `owner`：負責人可能缺失，所以也允許 `null`。

`["string", "null"]` 的意思是：這個欄位平常應該是文字，但如果原文沒有提供，可以用 `null` 表示未知。這比讓模型亂猜一個值更安全。


In [ ]:
event_text = "下週三 14:00 在 A203 教室有 AI 專題討論，負責人是小安。"

# 付費 API 示範預設關閉。要執行時，先確認 API key 與額度，再改成 True。
RUN_EVENT_EXTRACTION = False
if RUN_EVENT_EXTRACTION:
    # 讀這段時先分三層：event_text 是輸入資料，event_schema 是輸出契約，instructions 是抽取角色與限制。
    # structured_response() 只負責呼叫 API，回傳的仍是 response 物件。
    event_response = structured_response(
        event_text,
        event_schema,
        name="course_event",
        # instructions 補充任務語境；真正的欄位限制仍以 event_schema 為準。
        instructions="你是課程行政資料抽取助理。請抽取事件資訊。",
    )
    # parse_output_text() 才把 response.output_text 從 JSON 字串轉成 Python dict。
    event_data = parse_output_text(event_response)
    print_json(event_data, title="抽取結果")
else:
    print("事件抽取尚未執行；確認 API 成本後可將 RUN_EVENT_EXTRACTION 改為 True。")


### 🔍 輸出解析心智模型

Structured Outputs 讓 `response.output_text` 更容易是符合 schema 的 JSON 字串；Python 仍需用 `json.loads()` 轉成 dict，後續才能傳給表單、資料表或資料庫。

整個流程可以記成四步：

1. `structured_response()`：把原文、schema、instructions 送進 API。
2. API 回傳 `response`：裡面可能有 `output_text`，也可能有拒答或錯誤狀態。
3. `parse_output_text()`：先檢查 refusal / 空輸出，再用 `json.loads()` 轉成 Python dict。
4. `validate_event_data()`：確認 dict 的欄位符合 App 需要。

也就是說，Structured Outputs 幫你處理「模型輸出格式」，但你的程式仍要處理「錯誤情境」與「商業規則」。


---
## 3. 程式端驗證：不要只相信模型

Schema 能降低格式錯誤，但正式 App 仍應在程式端檢查必填欄位、重要欄位是否為空，以及是否有額外欄位進入系統。

這一節的驗證函式故意寫得很簡單，目的不是取代完整的 JSON Schema validator，而是讓你看懂正式 App 通常會有第二層防線：

- 第一層：API 依照 schema 產生資料。
- 第二層：Python 程式確認資料真的符合本系統要使用的條件。


In [ ]:
def validate_required_fields(data, required_fields):
    """
    檢查資料是否包含 schema 指定的 required 欄位。

    JSON Schema 的 required 代表「欄位必須出現在輸出中」，不代表欄位一定有非空值。
    例如 date 欄位如果允許 null，模型仍應輸出 date: null，而不是省略 date。
    這個函式只檢查欄位是否存在，把「欄位存在」和「欄位值是否可用」分開處理。

    參數 (Args):
        data:            已解析出的 dict。
        required_fields: schema 中 required 對應的欄位名稱清單。

    回傳 (Returns):
        錯誤訊息 list；若沒有錯誤，回傳空 list。
    """

    errors = []
    for field in required_fields:
        # 情況一：欄位不存在
        # 這代表模型輸出或測試資料沒有遵守 required 欄位清單。
        if field not in data:
            errors.append(f"缺少欄位：{field}")

        # 情況二：欄位存在但值是 None
        # 這裡不視為錯誤，因為某些欄位允許 null，代表原文沒有提供資訊。
    return errors


def validate_no_extra_fields(data, allowed_fields):
    """
    檢查資料是否出現 schema 沒有定義的額外欄位。

    在正式 App 中，額外欄位看似無害，但可能讓資料表、表單或後續處理流程變得不穩定。
    因此當 schema 使用 additionalProperties=False 時，程式端也應該保留相同檢查。

    參數 (Args):
        data:           已解析出的 dict。
        allowed_fields: schema 中 properties 允許的欄位名稱集合。

    回傳 (Returns):
        額外欄位的錯誤訊息 list；若沒有額外欄位，回傳空 list。
    """

    # set(data) 會取得 dict 的所有 key。
    # 如果 data 多出 allowed_fields 以外的 key，代表後續程式可能不知道怎麼處理。
    extras = sorted(set(data) - set(allowed_fields))
    return [f"不允許的欄位：{field}" for field in extras]


def validate_event_data(data):
    """
    針對 event_schema 執行本機端欄位驗證。

    Structured Outputs 已經要求模型遵守 schema，但正式應用仍建議在程式端再檢查一次。
    這個函式把 event_schema 裡的 required 與 properties 重新讀出來，
    避免 schema 修改後，驗證函式仍停留在舊欄位清單。

    參數 (Args):
        data: 已解析出的事件資料 dict。

    回傳 (Returns):
        所有欄位錯誤訊息的 list；若沒有錯誤，回傳空 list。
    """

    # 步驟一：從 schema 取出「必須出現」與「允許出現」的欄位清單。
    required = event_schema["required"]
    allowed = event_schema["properties"].keys()

    # 步驟二：合併兩種檢查結果。
    # 第一種檢查缺少 required 欄位；第二種檢查多出 schema 不允許的欄位。
    return validate_required_fields(data, required) + validate_no_extra_fields(data, allowed)


# 先用不用 API 的 sample_event 測試驗證函式，確認本機邏輯正確後再花錢呼叫模型。
sample_event = {"title": "AI 專題討論", "date": "下週三", "time": "14:00", "location": "A203 教室", "owner": "小安"}
print(validate_event_data(sample_event))


### ⚖️ Schema 驗證與商業規則驗證

JSON Schema 負責「格式」；你的程式仍要負責「商業規則」。例如日期是否真的存在、教室是否可借用、負責人是否為本班學生，這些通常需要額外資料來源。

簡單分法：

- **Schema 驗證**：有沒有 `title`？`needs_reply` 是不是 boolean？`items` 是不是 list？
- **商業規則驗證**：這間教室是否存在？金額是否和訂單系統一致？這個學生是否真的能交這份作業？

本章先做 schema 與基本欄位驗證，之後做 App 時再把商業規則接到資料庫或後端服務。


---
## 4. 陣列與巢狀資料：抽取多筆作業

前面的事件抽取只有一筆資料；作業說明通常會有多筆資料，所以需要 array。請注意：array 自己只代表「清單」，真正每一筆長什麼樣子，要寫在 `items` 裡。

閱讀 `assignment_schema` 時可以這樣拆：

- 最外層 object 只有一個欄位：`assignments`。
- `assignments` 是 array，代表多筆作業。
- `items` 定義每一筆作業都是 object。
- 每一筆作業都有 `name`、`due_date`、`submission`、`late_policy`。


In [ ]:
# assignment_schema 示範 array：一段文字裡可能有多筆作業。
assignment_schema = {
    "type": "object",
    "properties": {
        # 最外層只有 assignments 一個欄位；它的值是一個 list。
        "assignments": {
            "type": "array",
            "description": "文字中提到的作業清單",
            # items 定義 list 裡「每一筆作業」的格式。
            "items": {
                "type": "object",
                "properties": {
                    # 每筆作業至少要有 name；其餘欄位允許 null，保留「原文沒有交代」這個狀態。
                    "name": {"type": "string"},
                    "due_date": {"type": ["string", "null"]},
                    "submission": {"type": ["string", "null"]},
                    "late_policy": {"type": ["string", "null"]},
                },
                # array 裡每一筆作業也要有自己的 required 與 additionalProperties。
                "required": ["name", "due_date", "submission", "late_policy"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["assignments"],
    "additionalProperties": False,
}

assignment_text = """
作業一是 Prompt 改寫練習，9/30 前上傳 Moodle。
作業二是聊天機器人原型，10/15 23:59 前繳交 GitHub 連結，逾期不收 email 補交。
"""

print_json(assignment_schema, title="assignment_schema")


In [ ]:
# 這個示範會回傳巢狀資料：最外層 dict 裡有 assignments，assignments 裡才是一筆一筆作業。
RUN_ASSIGNMENT_EXTRACTION = False
if RUN_ASSIGNMENT_EXTRACTION:
    assignment_response = structured_response(
        assignment_text,
        assignment_schema,
        name="course_assignments",
        instructions="你是課程作業資訊抽取助理。請只抽取文字明確提供的資訊。",
    )
    assignment_data = parse_output_text(assignment_response)
    # parse 完後先看最外層 key，再看 list 裡每個 dict；這是閱讀巢狀 JSON 最穩的順序。
    # 預期 assignment_data["assignments"] 會是一個 list，每個元素是一筆作業 dict。
    print_json(assignment_data, title="作業抽取結果")
else:
    print("作業清單抽取尚未執行；確認 API 成本後可將 RUN_ASSIGNMENT_EXTRACTION 改為 True。")


### 🧩 陣列 Schema 的常見錯誤

- 忘記在 `items` 裡定義每一筆資料的 object 結構
- 忘記每層 object 都設定 `required`
- 允許太多額外欄位，導致後續程式難以處理
- 欄位名稱使用中文或空白，造成程式端不方便存取

看到巢狀 schema 時，不要一次看完整段。先用縮排找層級：最外層 object、裡面的 array、array 的每一個 item。每深入一層，就問：「這一層最後會變成 Python 的 dict 還是 list？」


---
## 5. 安全解析與錯誤處理

正式 App 不應假設 API 永遠成功，也不應假設 `output_text` 永遠能被 `json.loads()` 解析。

前面的 `structured_response()` 偏向「成功路徑」；這一節的 `extract_structured_safe()` 則是 App 比較常用的寫法：不讓錯誤直接中斷整個程式，而是回傳成功狀態與結果。

回傳格式設計成 `(ok, result)`：

- `ok == True` 時，`result` 是抽取成功的 dict。
- `ok == False` 時，`result` 是錯誤訊息，可以顯示在介面或記錄到 log。

這一節除了保留真正 API 呼叫的開關，也加入不花 API 費用的錯誤測試。錯誤測試會用假的 response 物件模擬三種情況：不是 JSON、沒有 `output_text`、模型拒答。這樣學生可以先看懂錯誤分流，再決定是否開啟付費 API 示範。


In [ ]:
def extract_structured_safe(user_input, schema, name, instructions=STRUCTURED_INSTRUCTIONS):
    """
    安全版 Structured Outputs 抽取流程。

    這個函式把 API 呼叫、拒答檢查、JSON 解析與例外處理包在一起。
    對 App 開發來說，這比讓例外一路拋到畫面層更容易控制使用者體驗：
    成功時回傳資料，失敗時回傳可顯示或記錄的錯誤訊息。

    參數 (Args):
        user_input:   要抽取資料的原始文字。
        schema:       JSON Schema dict。
        name:         schema / output format 的識別名稱。
        instructions: 給模型的任務說明。

    回傳 (Returns):
        (True, data): 成功時，data 是解析後的 dict。
        (False, error_message): 失敗時，error_message 是給開發者或使用者看的錯誤說明。
    """

    # App 程式通常不希望 API 錯誤直接讓整個畫面崩潰，所以這裡用 (ok, result) 明確分流成功與失敗。
    try:
        # 第一步：呼叫 API，要求模型依照 schema 輸出。
        response = structured_response(user_input=user_input, schema=schema, name=name, instructions=instructions)
        # 第二步：檢查 refusal / output_text，並把 JSON 字串轉成 Python dict。
        data = parse_output_text(response)
        # 第三步：用 ok=True 告訴呼叫端可以使用 data。
        return True, data
    except json.JSONDecodeError as exc:
        # 這類錯誤要保留原始訊息，方便回頭檢查 prompt、schema 或 SDK 回應格式。
        # JSONDecodeError 通常代表輸出不是合法 JSON，或 schema / SDK 行為需要檢查。
        return False, f"JSON 解析失敗：{exc}"
    except Exception as exc:
        # 其他錯誤可能是 API key、網路、模型拒答、schema 設計或 SDK 版本問題。
        return False, f"Structured Outputs 呼叫失敗：{exc}"


RUN_SAFE_EXTRACT_DEMO = False
if RUN_SAFE_EXTRACT_DEMO:
    ok, result = extract_structured_safe(event_text, event_schema, "course_event")
    # ok=True 時 result 是資料；ok=False 時 result 是錯誤訊息。
    print_json(result if ok else {"error": result})
else:
    print("安全抽取示範尚未執行；確認 API 成本後可將 RUN_SAFE_EXTRACT_DEMO 改為 True。")


class FakeResponse:
    """
    模擬 Responses API 回應，用來測試 parse_output_text() 的錯誤處理。

    這個 class 不會呼叫 API，只提供 output_text 與 output 兩個欄位。
    課堂上可以先用它確認錯誤分流，再開啟真正的 Structured Outputs 呼叫。
    """

    def __init__(self, output_text="", output=None):
        self.output_text = output_text
        self.output = output or []


parse_error_cases = {
    # 情況一：output_text 存在，但不是合法 JSON，會觸發 json.JSONDecodeError。
    "不是 JSON": FakeResponse(output_text="這不是 JSON"),
    # 情況二：output_text 是空字串，會被 parse_output_text() 擋下。
    "沒有 output_text": FakeResponse(output_text=""),
    # 情況三：模型拒答。即使 output_text 是空值，也要先看 refusal 原因。
    "模型拒答": FakeResponse(output=[{"content": [{"refusal": "這段內容不適合抽取。"}]}]),
}

print("本機錯誤解析測試：")
for label, fake_response in parse_error_cases.items():
    try:
        parse_output_text(fake_response)
        print(f"- {label}：未偵測到錯誤，請檢查測試案例。")
    except Exception as exc:
        print(f"- {label}：{type(exc).__name__} - {exc}")


### 🛑 無法完成時要怎麼處理？

Structured Outputs 仍可能遇到 API 錯誤、輸出被中斷、模型拒答或資料不足。正式應用至少要有友善錯誤訊息、保留原始輸入以便重試，並避免把不完整資料寫入資料庫。

特別注意：安全拒答不一定會符合你指定的 schema。程式端要先檢查 refusal 或空的 `output_text`，再做 `json.loads()`，不要假設每一次回應都能直接寫入資料庫。

實務上可以把錯誤分三類：

- **使用者輸入問題**：輸入太短、缺少必要資訊、和任務無關。
- **模型或 API 問題**：拒答、回應中斷、暫時性服務錯誤。
- **程式規格問題**：schema 寫錯、欄位名稱不一致、Pydantic 限制太嚴或太鬆。

retry 只能處理少數暫時性 API 問題，不能修好設計不良的 schema。


---
## 6. 批次測試：確認 Schema 是否穩定

單筆成功不代表 Schema 穩定。至少要準備幾筆不同格式的文字，觀察欄位是否一致。

批次測試不是為了追求每一次都完美，而是幫你找出 schema 哪裡太模糊。建議測試資料至少包含：

- 資訊完整的正常案例。
- 資訊缺漏但仍可接受的案例，例如沒有負責人時輸出 `owner: null`。
- 明確應該失敗的錯誤案例，例如缺少 required 欄位，或多出 schema 不允許的欄位。
- 表達方式不同的案例，例如「晚上 7 點」、「線上舉辦」。

如果多筆資料都在同一欄出錯，優先修改 schema 的欄位描述或欄位型別，再考慮調整 prompt。


In [ ]:
event_test_cases = [
    "本週五 10:00 在 B101 開會，主題是期中專題檢查，負責人小華。",
    "明天下午到圖書館討論 Streamlit 部署，沒有指定負責人。",
    "10/20 晚上 7 點線上舉辦 GitHub 操作複習，由助教阿明帶領。",
]


def run_event_batch_tests(cases):
    """
    使用同一份 event_schema 批次測試多段事件文字。

    單一範例成功不代表 schema 穩定。批次測試可以讓學生觀察不同寫法、缺漏資訊、
    日期格式或負責人描述方式是否仍能被抽成一致欄位。這也是把 notebook 範例推向 App 前的基本測試習慣。

    參數 (Args):
        cases: 多段事件描述文字的 list。

    回傳 (Returns):
        測試結果 list。每筆結果包含 case 編號、ok 狀態、data 或錯誤訊息，以及欄位驗證 errors。
    """

    results = []
    for index, text in enumerate(cases, start=1):
        # 每一筆測試都走同一個 schema 與同一組 validation，才能比較穩定性。
        ok, data = extract_structured_safe(
            text,
            event_schema,
            name="course_event",
            instructions="你是課程活動資料抽取助理。",
        )
        # API 成功後仍做程式端欄位檢查；API 失敗時不再驗證 data。
        errors = [] if not ok else validate_event_data(data)
        # 每筆結果都保留原始 data 與 errors；課堂上才能比較「模型抽對但驗證失敗」和「API 呼叫失敗」的差別。
        results.append({"case": index, "ok": ok and not errors, "data": data, "errors": errors})
    return results


RUN_EVENT_BATCH_TESTS = False
if RUN_EVENT_BATCH_TESTS:
    print_json(run_event_batch_tests(event_test_cases), title="批次測試結果")
else:
    print("批次測試尚未執行；確認 API 成本後可將 RUN_EVENT_BATCH_TESTS 改為 True。")


local_validation_cases = [
    (
        "正確資料",
        {"title": "AI 專題討論", "date": "下週三", "time": "14:00", "location": "A203 教室", "owner": "小安"},
    ),
    (
        "缺少 required 欄位 owner",
        {"title": "AI 專題討論", "date": "下週三", "time": "14:00", "location": "A203 教室"},
    ),
    (
        "多出 schema 不允許的欄位 status",
        {
            "title": "AI 專題討論",
            "date": "下週三",
            "time": "14:00",
            "location": "A203 教室",
            "owner": "小安",
            "status": "confirmed",
        },
    ),
]


def run_local_validation_tests(cases):
    """
    不呼叫 API，直接用手寫 dict 測試 validate_event_data()。

    批次 API 測試主要觀察模型抽取是否穩定；本機 validation 測試則故意放入錯誤資料，
    確認程式端真的會擋下缺少 required 欄位或多出未允許欄位的結果。
    """

    results = []
    for label, data in cases:
        errors = validate_event_data(data)
        results.append({"case": label, "ok": not errors, "errors": errors, "data": data})
    return results


print_json(run_local_validation_tests(local_validation_cases), title="本機欄位驗證測試")


### 🔁 Schema 改版紀錄與進階路線

當你發現抽取結果不穩定時，不要只改 prompt。請同時記錄 schema 版本、改了什麼，以及為什麼。Structured Outputs 的穩定性不只來自模型，也來自你給它的資料規格是否清楚。

本週主線先使用手寫 JSON Schema，因為它最能看懂 Structured Outputs 的底層結構：欄位名稱、資料型別、必填欄位、巢狀物件與陣列。接著補充 Pydantic 版本，讓後續 Python App 可以用類別定義同一份資料格式，並用 `client.responses.parse()` 直接取得已驗證的 Python 物件。

版本紀錄的重點不是形式，而是讓之後的你知道「為什麼當初這樣設計」。例如前面事件抽取的 schema 可以這樣演進：

- v1：先抽 `title/date/time/location/owner`，確認最小可用資料。
- v2：新增 `event_type`，因為 App 需要區分會議、作業、提醒。
- v3：把 `date` 從原文改成 ISO 日期，因為要接 Google Calendar 或資料庫查詢。

下面的 Pydantic 範例也可以用同樣方式理解，不是突然換一個無關技術：

- v1：`ReviewInsight` 先保留 `customer_name` 和 `summary`，確認能從評論抽出基本資訊。
- v2：加入 `rating: int = Field(ge=1, le=5)`，因為 App 後面可能要計算平均分數，不能讓「五顆星」停留在文字。
- v3：加入 `sentiment: Literal["positive", "neutral", "negative"]`，因為分類結果要進入圖表或篩選器，不能每次出現不同說法。
- v4：加入 `MeetingMinutes` 與 `action_items: list[ActionItem]`，因為會議記錄通常不是單一欄位，而是一份資料裡有多筆待辦事項。

因此，JSON Schema 與 Pydantic 的關係可以這樣看：JSON Schema 幫你看懂結構化輸出的底層規格；Pydantic 把同一份規格改寫成 Python App 更容易維護、驗證與重用的 class。


In [ ]:
# schema_registry 用來記錄目前可用的 schema 版本。
# deepcopy() 可避免後面不小心修改 registry 裡的原始 schema。
schema_registry = {
    # 用版本號命名，日後欄位異動時可以新增 v2，而不是直接覆蓋正在使用的規格。
    "course_event_v1": deepcopy(event_schema),
    "course_assignments_v1": deepcopy(assignment_schema),
}


def list_schema_names(registry):
    """
    列出目前 schema registry 中已登錄的 schema 名稱。

    當一個 App 同時有事件抽取、作業抽取、客服分類等多種格式時，
    用 registry 管理 schema 會比散落在多個變數中更容易維護。

    參數 (Args):
        registry: schema 名稱到 schema dict 的對照表。

    回傳 (Returns):
        排序後的 schema 名稱 list，方便顯示或比對。
    """

    return sorted(registry.keys())


def get_schema(registry, name):
    """
    從 schema registry 取出指定 schema 的副本。

    這個函式不直接回傳 registry 內部的原始物件，而是使用 deepcopy() 建立副本。
    原因是 dict/list 是可變物件；如果呼叫端拿到原始 schema 後不小心修改，
    可能會影響之後所有使用同一份 registry 的抽取流程。

    參數 (Args):
        registry: schema 名稱到 schema dict 的對照表。
        name:     想要取出的 schema 名稱。

    回傳 (Returns):
        指定 schema 的深拷貝副本。

    可能拋出 (Raises):
        KeyError: registry 中找不到指定名稱。
    """

    # 情況一：名稱不存在
    # 直接拋出 KeyError，讓呼叫端知道不是 API 問題，而是 schema 名稱打錯或尚未登錄。
    if name not in registry:
        raise KeyError(f"找不到 schema：{name}")

    # 情況二：名稱存在
    # 回傳 deepcopy，保護 registry 裡的正式版本不被外部程式改壞。
    return deepcopy(registry[name])


print("目前 schema：", list_schema_names(schema_registry))

# 實際取用其中一份 schema，示範 registry 不只是列清單，也能提供後續 API 呼叫要用的規格。
# 這裡先不呼叫 API，只檢查取出的 schema 名稱與欄位，避免學生誤以為 get_schema() 沒有用途。
selected_schema_name = "course_event_v1"
selected_schema = get_schema(schema_registry, selected_schema_name)
print("已取出 schema：", selected_schema_name)
print("schema 欄位：", list(selected_schema["properties"].keys()))


---
## 7. Pydantic 進階：用類別定義輸出格式

Pydantic 是 Python 常用的資料驗證與資料模型工具。它不是新的 AI 模型，也不是資料庫；它的角色是把「一筆資料應該長什麼樣子」寫成 Python class，並在程式收到資料時檢查欄位是否符合規格。

在本週情境中，Pydantic 解決的是這個問題：模型回傳的內容即使看起來像 JSON，App 仍然需要確認欄位名稱、型別、範圍與固定選項是否正確。手寫 JSON Schema 讓你看懂底層規格；Pydantic 讓你在 Python 專案裡用 class 管理同一份規格。

最小的 Pydantic 模型長這樣：

```python
class ReviewInsight(BaseModel):
    customer_name: str
    rating: int = Field(ge=1, le=5)
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str
```

這段 class 的意思是：抽取結果必須有顧客姓名、1 到 5 的整數評分、固定三選一的情緒標籤，以及一句摘要。後面的程式把這個 class 傳給 `parse_with_pydantic(text, ReviewInsight)`，也就是告訴 API：「請依照 `ReviewInsight` 這份資料合約回傳結果。」

可以把兩種寫法這樣對照：

| JSON Schema | Pydantic | 意思 |
|-------------|----------|------|
| `"type": "string"` | `name: str` | 欄位必須是文字 |
| `"type": "integer"` | `quantity: int` | 欄位必須是整數 |
| `"enum": [...]` | `Literal[...]` | 只能從固定選項中選一個 |
| `"minimum": 1` | `Field(ge=1)` | 數字至少為 1 |
| `"maximum": 5` | `Field(le=5)` | 數字最多為 5 |
| `"items": {...}` | `list[Model]` | 清單裡每一筆都有固定結構 |

| Pydantic 寫法 | 在本週範例中的用途 |
|---------------|----------------------|
| `BaseModel` | 每個資料模型都繼承它，例如 `ReviewInsight`、`ActionItem`、`MeetingMinutes` |
| `Field(ge=1, le=5)` | 限制評分、數量或金額範圍，避免不合理資料進入 App |
| `Literal["a", "b"]` | 限制分類結果只能是固定標籤，類似 JSON Schema 的 `enum` |
| `Optional[str] = None` | 欄位可能沒有明確出現，例如待辦事項沒有期限 |
| `list[ActionItem]` | 表示一份會議記錄裡有多筆待辦事項，對應 JSON Schema 的 array / items |
| `ValidationError` | 當欄位型別或限制不符合規格時，用來捕捉錯誤 |

讀 Pydantic class 時，先看「欄位名稱」，再看冒號後面的「型別」，最後才看 `Field()`、`Literal` 或 `Optional` 這些限制。這樣會比從 `BaseModel` 或 import 開始看更容易。

請把下一個 code cell 分成三層讀：

1. `ReviewInsight`、`ActionItem`、`MeetingMinutes`：定義資料長相。
2. `parse_with_pydantic()`：把 Pydantic class 交給 `client.responses.parse()`，取得 `response.output_parsed`。
3. `extract_structured()`：把驗證失敗、API 失敗等錯誤集中處理，方便之後接 Streamlit 或 FastAPI。

> 若課堂環境的 SDK 尚未支援 `responses.parse()`，請先回到前面的 `text.format` + `json.loads()` 主線。Pydantic 是進階維護路線，不取代本週的 JSON Schema 基礎。


In [ ]:
# Pydantic 範例：評論資料抽取
# 這一段和前面的 JSON Schema 做同一件事：定義模型輸出資料的欄位規格。
# 差別是 Pydantic 用 Python class 寫，後續 App 維護時通常比較容易讀。
review_text = "王小明給了這家餐廳五顆星，他說：食物非常好吃，服務也很親切，會再來！"


class ReviewInsight(BaseModel):
    """
    餐廳評論抽取結果的資料模型。

    學生要在這個 class 中補上評分、情緒與摘要欄位。
    寫 TODO 時先想清楚每個欄位的資料型態，再決定是否需要 Field 或 Literal 驗證。
    """

    # 每一行都是一個欄位：欄位名稱在冒號左邊，型別在冒號右邊。
    # 先從最容易確定的 customer_name 開始，再補 rating、sentiment、summary；這樣能把欄位命名和驗證規則分開思考。
    customer_name: str
    # TODO 1：新增 rating 欄位，型別 int，限制範圍 1~5（提示：Field(ge=1, le=5)）
    # TODO 2：新增 sentiment 欄位，只能是 "positive" / "neutral" / "negative"（提示：Literal）
    # TODO 3：新增 summary 欄位，型別 str，代表一句話摘要


class ActionItem(BaseModel):
    """
    會議記錄中的一筆待辦事項。

    這個模型會被 MeetingMinutes 巢狀使用，對應 JSON Schema 裡 array items 的概念。
    每個 action item 都要有負責人與任務；期限可能沒有明確出現，所以 due_date 允許 None。
    """

    # 一筆待辦事項包含負責人、任務，以及可能沒有明確日期的 due_date。
    owner: str
    task: str
    due_date: Optional[str] = None


class MeetingMinutes(BaseModel):
    """
    會議記錄抽取結果的完整資料模型。

    attendees 是純文字 list，action_items 則是 list[ActionItem]。
    這讓學生看到 Pydantic 如何表示「一份資料裡包含多筆子資料」的巢狀結構。
    """

    # 巢狀模型：action_items 是一個 list，list 裡每一筆都必須符合 ActionItem。
    meeting_topic: str
    attendees: list[str]
    action_items: list[ActionItem]


def parse_with_pydantic(text, model_cls, instructions="你是嚴謹的資料抽取助理，只依照提供文字填寫欄位。"):
    """
    使用 responses.parse() 直接取得 Pydantic 物件。

    前半段課程使用手寫 JSON Schema，目的是讓學生理解欄位契約如何運作。
    這裡改用 Pydantic class 描述同樣的資料契約，SDK 會依據 class 產生結構化輸出規格，
    並把結果解析成有型別的 Python 物件。這種寫法適合欄位較多、需要 Field / Literal 驗證的 App。

    參數 (Args):
        text:         要抽取資料的原始文字。
        model_cls:    Pydantic 模型類別，例如 ReviewInsight 或 MeetingMinutes。
        instructions: 給模型的任務說明，通常用來強調只依據原文、不編造。

    回傳 (Returns):
        response.output_parsed，也就是已通過 Pydantic 驗證的模型物件。
    """

    # responses.parse() 會根據 Pydantic class 產生結構化輸出規格，適合欄位較多或需要驗證規則的 App。
    # 讀這段時把 model_cls 想成「資料合約」：傳入 ReviewInsight 就抽評論，傳入 MeetingMinutes 就抽會議記錄。
    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=text,
        # text_format 接收的是 Pydantic class，不是手寫 JSON Schema dict；SDK 會處理格式約束與 parsed 結果。
        text_format=model_cls,
    )
    # output_parsed 已經是 Pydantic 物件，不需要再 json.loads()。
    return response.output_parsed


def extract_structured(text, model_cls, instructions="你是嚴謹的資料抽取助理，只依照提供文字填寫欄位，不要編造。"):
    """
    App 可重用的 Pydantic 抽取 wrapper。

    parse_with_pydantic() 適合示範核心 API 用法，但正式 App 通常還需要處理驗證失敗、
    API 失敗或模型拒答。這個 wrapper 把錯誤處理集中在一個地方，
    讓 Streamlit 或 FastAPI 頁面只要判斷回傳值是否為 None，就能決定要顯示結果還是錯誤提示。

    參數 (Args):
        text:         要抽取資料的原始文字。
        model_cls:    Pydantic 模型類別。
        instructions: 給模型的任務說明。

    回傳 (Returns):
        成功時回傳 Pydantic 物件；失敗時印出錯誤並回傳 None。
    """

    # wrapper 把錯誤處理集中在一個地方；Streamlit 或 FastAPI 頁面只要判斷回傳是否為 None。
    try:
        return parse_with_pydantic(text, model_cls, instructions=instructions)
    except ValidationError as exc:
        # 欄位型別、Literal 選項或 Field 範圍不符合時，會走到這裡。
        print("欄位驗證失敗：", exc)
        return None
    except Exception as exc:
        # API key、網路、SDK 版本或模型拒答等問題，會走到這裡。
        print("Structured parsing 失敗：", exc)
        return None


meeting_text = """
今天的產品會議由小美主持，出席者有小美、志明、雅婷。
決議事項：
1. 志明要在下週五前完成新版首頁設計。
2. 雅婷負責整理使用者訪談紀錄，沒有明確期限。
3. 小美會發送會議記錄給全體團隊，今天之內完成。
"""

# Pydantic 範例仍預設關閉，因為它會真正呼叫 API；完成欄位定義後再開啟比較容易除錯。
RUN_PYDANTIC_REVIEW_DEMO = False
if RUN_PYDANTIC_REVIEW_DEMO:
    insight = extract_structured(
        "請從以下評論抽取顧客姓名、評分、情緒與一句話摘要：\n\n" + review_text,
        ReviewInsight,
    )
    print(insight)
else:
    print("Pydantic review 示範尚未執行；完成 ReviewInsight 後再改為 True。")

RUN_MEETING_MINUTES_DEMO = False
if RUN_MEETING_MINUTES_DEMO:
    minutes = extract_structured(
        "請從以下會議記錄抽取會議主題、出席者與待辦事項：\n\n" + meeting_text,
        MeetingMinutes,
    )
    print(minutes)
else:
    print("巢狀 MeetingMinutes 示範尚未執行；確認 API 成本後可改為 True。")


---
## 8. 課堂練習 A：客服訊息資料抽取器

請用手寫 JSON Schema 完成客服訊息抽取器。這題延續前半段主線，重點是練習 enum、boolean 與必要欄位。

解題順序建議：

1. 先決定 App 後面需要讀哪些欄位。
2. 在 `properties` 裡為每個欄位寫 `type`。
3. 如果值只能是固定幾種，就加 `enum`。
4. 把每個欄位列進 `required`。
5. 設定 `additionalProperties: False`，避免模型多生欄位。


In [ ]:
# 練習 A：客服訊息資料抽取器
# 先完成 properties，再完成 required 和 additionalProperties。
# properties 是欄位規格；required 是欄位清單，兩者都要寫才完整。
support_schema = {
    "type": "object",
    "properties": {
        # TODO 1：定義 category，建議 enum: 功能問題、帳務問題、操作疑問、其他
        # category 是後續分派客服流程會用到的分類欄位，所以建議用 enum 限制固定選項。
        # TODO 2：定義 urgency，建議 enum: low、medium、high
        # urgency 不要只看情緒字眼，也要看時間壓力與業務影響，例如「今天處理」通常不是 low。
        # TODO 3：定義 summary，型別為 string
        # summary 應該是給客服快速掃描用的短句，不要把整段原文複製回來。
        # TODO 4：定義 needs_reply，型別為 boolean
        # needs_reply 用 boolean，是因為程式後續通常會用 True/False 決定是否進入回覆佇列。
    },
    # TODO 5：列出 required 欄位
    # required 建議包含所有 App 會依賴的欄位；若資訊未知，可讓欄位值表達未知，而不是讓欄位消失。
    # TODO 6：設定 additionalProperties 為 False
    # additionalProperties=False 可以防止模型多生欄位，降低後續資料表或 UI 處理成本。
}

support_message = "我昨天被扣款兩次，後台訂閱狀態還顯示未啟用，請今天幫我處理。"

# TODO 7：呼叫 extract_structured_safe()，印出抽取結果
# 建議先把 name 設成 support_ticket，instructions 說明你要分類客服訊息與判斷是否需要回覆。
# 提醒：如果 ok 是 False，請印出錯誤訊息；如果 ok 是 True，請印出抽取資料。


---
### 練習 B：收據資料抽取器

請用 Pydantic 定義訂單資料格式，抽出訂單編號、顧客、商品清單與運費。這題展示結構化資料抽取後，可以直接進行小計與總額計算。

這題的閱讀重點是巢狀資料：`OrderInfo` 裡面有 `items: list[OrderItem]`，意思是訂單有多個商品，而每個商品都要符合 `OrderItem` 的欄位規格。

完成後請檢查兩件事：

- 商品數量與單價是否是整數，才能計算小計。
- 運費是否被抽成數字，才能加到總額。


In [ ]:
receipt_text = """
訂單編號 A1023，顧客：陳建宏。
購買項目：
- 藍牙耳機 x2，單價 990 元
- 保護殼 x1，單價 250 元
運費 60 元。
"""


class OrderItem(BaseModel):
    """
    訂單中的一筆商品資料。

    學生要把商品名稱、數量與單價拆成可計算欄位。
    這個 class 會被 OrderInfo.items 使用，所以欄位型態會直接影響後面的總額計算。
    """

    # 一筆商品資料。只有數量和單價是數字，後面才能做乘法計算。
    # 這裡不要把「x2」或「990 元」整串留成文字，否則後面的 subtotal 會很難算。
    item_name: str
    # TODO 1：新增 quantity，型別 int，限制至少 1
    # TODO 2：新增 unit_price，型別 int，限制至少 0


class OrderInfo(BaseModel):
    """
    收據 / 訂單抽取結果的完整資料模型。

    學生要在這裡補上顧客姓名、商品清單與運費。
    完成後，parse_with_pydantic() 回傳的 order 物件就能直接拿來計算 subtotal 與總額。
    """

    # 一張訂單資料。items 是多筆商品，所以型別會是 list[OrderItem]。
    # 這和前面的 JSON Schema array/items 是同一個概念，只是改用 Python 型別表示。
    order_id: str
    # TODO 3：新增 customer_name，型別 str
    # TODO 4：新增 items，型別 list[OrderItem]
    # TODO 5：新增 shipping_fee，型別 int，限制至少 0


# TODO 6：使用 parse_with_pydantic(receipt_text, OrderInfo) 抽取資料
# 這一步若成功，order 會是 OrderInfo 物件；存取欄位時使用 order.items、order.shipping_fee。
# TODO 7：若成功，計算商品小計與加運費後總額
# 提醒：商品小計可以用 sum(item.quantity * item.unit_price for item in order.items)。


In [ ]:
def extract_with_retry(text, model_cls, retries=2):
    """
    對 Pydantic structured parsing 加上有限次數的重試。

    重試適合處理暫時性 API 或網路問題，但不應該拿來掩蓋 schema 設計錯誤。
    因此這個函式會把 ValidationError 和一般 Exception 分開顯示，讓學生知道：
    欄位限制不合理時，應該修 schema 或輸入資料；服務暫時失敗時，才比較適合重試。

    參數 (Args):
        text:      要抽取資料的原始文字。
        model_cls: Pydantic 模型類別。
        retries:   失敗後最多重試幾次；總嘗試次數會是 retries + 1。

    回傳 (Returns):
        成功時回傳 Pydantic 物件；全部嘗試失敗時回傳 None。
    """

    # 這題的目標是把前面的 parse_with_pydantic() 包成比較耐用的函式。
    # 注意 retries=2 代表失敗後最多重試 2 次，所以總嘗試次數是 3 次。
    # TODO 1：用迴圈呼叫 parse_with_pydantic(text, model_cls)，最多嘗試 retries + 1 次
    # attempt 從 1 開始比較適合印給使用者看；range 的結束值要多 1 才會包含最後一次嘗試。
    # TODO 2：捕捉 ValidationError 與一般 Exception，印出第幾次失敗
    # ValidationError 偏向資料格式不符；一般 Exception 可能是 API key、網路或服務暫時失敗。
    # TODO 3：只要有一次成功就直接回傳結果
    # TODO 4：全部失敗時回傳 None
    pass


# TODO 5：用 ReviewInsight 或 OrderInfo 測試 extract_with_retry()


---
## ✅ 完成檢核

- [ ] 能說明 Structured Outputs 與一般 JSON prompt 的差異
- [ ] 能寫出包含 `type`、`properties`、`required`、`additionalProperties` 的 schema
- [ ] 能看懂 array schema 中 `items` 代表「每一筆資料的格式」
- [ ] 能用 `text.format` 呼叫 Structured Outputs
- [ ] 能將 `response.output_text` 解析成 Python dict
- [ ] 能處理模型拒答、空輸出與 JSON 解析失敗
- [ ] 能做基本必填欄位與額外欄位檢查
- [ ] 能區分 schema 驗證與商業規則驗證
- [ ] 能用 Pydantic 定義欄位限制與巢狀資料
- [ ] 能完成至少一個自己的資料抽取器

如果以上有任何一項不確定，請回到對應章節，只看該章的「輸入文字、輸出規格、程式檢查」三層，不需要一次理解整份 notebook。


## 📝 課後任務與延伸閱讀

請選擇你的期中專題，設計一個資料抽取器：找一段真實或模擬輸入、設計 JSON Schema、用 Structured Outputs 抽取資料、做至少三筆測試，並記錄 schema v1 到 v2 的修改理由。

建議紀錄格式：

| 項目 | 內容 |
|------|------|
| 輸入來源 | 例如客服訊息、收據、活動公告、課程通知 |
| v1 schema | 第一版欄位設計 |
| 測試結果 | 至少三筆輸入的成功與失敗狀況 |
| v2 修改 | 根據失敗原因調整欄位、型別或描述 |
| App 用途 | 抽出的資料要放到表格、表單、提醒、統計或搜尋 |

延伸閱讀：
- OpenAI Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- Responses API reference: https://platform.openai.com/docs/api-reference/responses
- JSON Schema: https://json-schema.org/learn/getting-started-step-by-step

下週會銜接 Streamlit 與部署流程，把本週抽取出的 dict 放進可操作的網頁介面。
